# LASSER on Kaggle

GEARS with an optional GSR-style learned co-expression graph. Needs **Internet on** (GEARS downloads Norman and the GO graph from Harvard Dataverse) and a **GPU (T4)**.

Cells: install, get the code, run the tests (before any real training), baseline run, graph-learning run, results.

In [1]:
# 1. Dependencies. GEARS is vendored inside lasser/ (lasser/gears), so no cell-gears install.
#    Kaggle already has torch, numpy, pandas, scipy, scikit-learn, networkx, tqdm, requests.
!pip install -q torch_geometric scanpy pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00


In [2]:
# 2. Get the code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"   # <- set this
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

02a13c9 (HEAD -> main, origin/main, origin/HEAD) fix: format issue fixing


In [3]:
# 3. Tests 1-8 (tiny subsets; first run also downloads Norman, ~10-20 min in total).
%env LASSER_DATA_DIR=/kaggle/working/data
%env LASSER_TEST_DIR=/kaggle/working/lasser_tests
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/test_output.txt | grep -vE "it/s\]|s/it\]"

env: LASSER_DATA_DIR=/kaggle/working/data
env: LASSER_TEST_DIR=/kaggle/working/lasser_tests
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 16 items

lasser-repo/tests/test_00_packaging.py ..lasser modules loaded: ['lasser', 'lasser._eval', 'lasser.config', 'lasser.evaluation', 'lasser.gears', 'lasser.gears.data_utils', 'lasser.gears.gears', 'lasser.gears.inference', 'lasser.gears.model', 'lasser.gears.pertdata', 'lasser.gears.utils', 'lasser.gears.version', 'lasser.run', 'lasser.tracking', 'lasser.utils']
.
lasser-repo/tests/test_01_parity.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'R

In [4]:
# 3b. Optional test 9: full Norman, 1 epoch, flag off and on, checks the T4 memory budget.
%env LASSER_SMOKE=1
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_09_smoke.py -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/smoke_output.txt | grep -vE "it/s\]|s/it\]"
%env LASSER_SMOKE=0

env: LASSER_SMOKE=1
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 2 items

lasser-repo/tests/test_09_smoke.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
2026-10-06 14:16:26,460 INFO lasser.tracking: run f3006654 -> /kaggle/working/lasser_tests/runs/norman_seed1_base_20261006-141626_smoke
2026-10-06 14:16:26,528 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training)
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:1

In [5]:
# 4. Load the data once and reuse it for both runs.
from lasser import GSRConfig, LasserConfig, load_pert_data, load_run, run_experiment

common = dict(dataset="norman", seed=1, epochs=1,
              data_dir="/kaggle/working/data", out_dir="/kaggle/working/runs",
              cache_dir="/kaggle/working/cache", repo_dir=REPO_DIR)
pert_data = load_pert_data(LasserConfig(**common))

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!


In [6]:
# 5. Baseline: graph_learning=False is plain GEARS (plus logging and saving).
base = run_experiment(LasserConfig(**common, graph_learning=False), pert_data)
base["run_dir"]

2026-10-06 14:35:27,974 INFO lasser.tracking: run 87d0540f -> /kaggle/working/runs/norman_seed1_base_20261006-143527
2026-10-06 14:35:28,033 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training)


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 14:35:28,049 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75.pkl', 'norman_simulation_1_0.75_subgroup.pkl'])
2026-10-06 14:35:28,052 INFO lasser.utils: [gears_init] start


Found local copy...


2026-10-06 14:35:47,451 INFO lasser.utils: [gears_init] done in 19.4s, peak CUDA memory 0.03 GB
2026-10-06 14:35:47,456 INFO lasser.run: co-expression graph used (gears_static): {'num_nodes': 5045, 'num_edges': 6364, 'num_self_loops': 4699, 'degree_min': 0, 'degree_median': 0.0, 'degree_mean': 0.33002973240832506, 'degree_max': 20, 'degree_gini': 0.9324442777762897, 'isolated_genes': 4419}
2026-10-06 14:35:47,457 INFO lasser.run: training GEARS on cuda:0 (Tesla T4, 14.6 GB) for 1 epochs
2026-10-06 14:35:47,458 INFO lasser.utils: [gears_train] start


Start Training...
Epoch 1 Step 1 Train Loss: 0.5244
Epoch 1 Step 51 Train Loss: 0.6099
Epoch 1 Step 101 Train Loss: 0.5167
Epoch 1 Step 151 Train Loss: 0.5267
Epoch 1 Step 201 Train Loss: 0.4575
Epoch 1 Step 251 Train Loss: 0.5969
Epoch 1 Step 301 Train Loss: 0.4178
Epoch 1 Step 351 Train Loss: 0.5540
Epoch 1 Step 401 Train Loss: 0.5095
Epoch 1 Step 451 Train Loss: 0.5325
Epoch 1 Step 501 Train Loss: 0.4450
Epoch 1 Step 551 Train Loss: 0.6025
Epoch 1 Step 601 Train Loss: 0.4668
Epoch 1 Step 651 Train Loss: 0.5007
Epoch 1 Step 701 Train Loss: 0.4617
Epoch 1 Step 751 Train Loss: 0.4461
Epoch 1 Step 801 Train Loss: 0.4754
Epoch 1 Step 851 Train Loss: 0.4256
Epoch 1 Step 901 Train Loss: 0.5404
Epoch 1 Step 951 Train Loss: 0.4250
Epoch 1 Step 1001 Train Loss: 0.4224
Epoch 1 Step 1051 Train Loss: 0.4816
Epoch 1 Step 1101 Train Loss: 0.5083
Epoch 1 Step 1151 Train Loss: 0.4291
Epoch 1 Step 1201 Train Loss: 0.4356
Epoch 1 Step 1251 Train Loss: 0.4418
Epoch 1 Step 1301 Train Loss: 0.4841
Epoch 

2026-10-06 14:41:28,266 INFO lasser.utils: [gears_train] done in 340.8s, peak CUDA memory 0.87 GB
2026-10-06 14:41:28,290 INFO lasser.utils: [evaluate] start
2026-10-06 14:44:30,497 INFO lasser.evaluation: test (eval.py): pearson_delta=0.3604 mse_top20_de=0.3465 nmse_top20_de=3.9660 over 107 perts
2026-10-06 14:44:30,498 INFO lasser.evaluation:   combo_seen0    pearson_delta=0.3925 mse_top20_de=0.3958 (9 perts)
2026-10-06 14:44:30,499 INFO lasser.evaluation:   combo_seen1    pearson_delta=0.3845 mse_top20_de=0.3799 (43 perts)
2026-10-06 14:44:30,499 INFO lasser.evaluation:   combo_seen2    pearson_delta=0.4038 mse_top20_de=0.4010 (19 perts)
2026-10-06 14:44:30,501 INFO lasser.evaluation:   unseen_single  pearson_delta=0.3008 mse_top20_de=0.2656 (36 perts)
2026-10-06 14:44:30,503 INFO lasser.utils: [evaluate] done in 182.2s, peak CUDA memory 1.15 GB
2026-10-06 14:44:30,529 INFO lasser.tracking: appended run 87d0540f to /kaggle/working/runs/runs.csv


'/kaggle/working/runs/norman_seed1_base_20261006-143527'

In [7]:
# 6. Graph learning: GSR-refined co-expression graph (defaults from the plan, section 11).
gl = run_experiment(LasserConfig(**common, graph_learning=True, gsr=GSRConfig()), pert_data)
gl["run_dir"], gl["checks"]["h2"], gl["checks"]["refine"]

2026-10-06 14:44:30,564 INFO lasser.tracking: run 348393ed -> /kaggle/working/runs/norman_seed1_gl_20261006-144430
2026-10-06 14:44:30,573 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training and GSR graph learning)


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 14:44:30,589 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75.pkl', 'norman_simulation_1_0.75_subgroup.pkl'])
2026-10-06 14:44:30,591 INFO lasser.utils: [gsr_graph] start
2026-10-06 14:44:30,644 INFO lasser.graph: GSR graph learning on cuda:0 (Tesla T4, 14.6 GB)
2026-10-06 14:44:30,644 INFO lasser.utils: [gsr/data] start
2026-10-06 14:44:31,202 INFO lasser.data: training cells: 49849 (A0 subset 38824, H1 24957, H2 24892) over 138 training perturbations
2026-10-06 14:44:31,203 INFO lasser.utils: [gsr/data] done in 0.6s, peak CUDA memory 0.05 GB
2026-10-06 14:44:31,204 INFO lasser.utils: [gsr/views] start
2026-10-06 14:44:41,634 INFO lasser.data: metacells: 49849 cells -> 500 metacells
2026-10-06 14:44:42,540 INFO lasser.views: view E: dim 64, missing genes 234
2026-10-06 14:44:43,043 INFO lasser.views: view R: dim 32, missing genes 271
2026-10-06 14:44:43,540 INFO lasser.views: deepwalk epoch 1: loss 1.6828 over 1064200 pairs
2026-10-

Found local copy...


2026-10-06 14:45:15,146 INFO lasser.utils: [gears_init] done in 21.3s, peak CUDA memory 0.08 GB
2026-10-06 14:45:15,150 INFO lasser.run: co-expression graph used (gsr_refined): {'num_nodes': 5045, 'num_edges': 45192, 'num_self_loops': 4699, 'degree_min': 5, 'degree_median': 7.0, 'degree_mean': 8.026362735381566, 'degree_max': 25, 'degree_gini': 0.20115421336879258, 'isolated_genes': 0}
2026-10-06 14:45:15,246 INFO lasser.run: training GEARS on cuda:0 (Tesla T4, 14.6 GB) for 1 epochs
2026-10-06 14:45:15,247 INFO lasser.utils: [gears_train] start


Start Training...
Epoch 1 Step 1 Train Loss: 0.5244
Epoch 1 Step 51 Train Loss: 0.6158
Epoch 1 Step 101 Train Loss: 0.5137
Epoch 1 Step 151 Train Loss: 0.5235
Epoch 1 Step 201 Train Loss: 0.4555
Epoch 1 Step 251 Train Loss: 0.5953
Epoch 1 Step 301 Train Loss: 0.4119
Epoch 1 Step 351 Train Loss: 0.5401
Epoch 1 Step 401 Train Loss: 0.5129
Epoch 1 Step 451 Train Loss: 0.5277
Epoch 1 Step 501 Train Loss: 0.4399
Epoch 1 Step 551 Train Loss: 0.5978
Epoch 1 Step 601 Train Loss: 0.4604
Epoch 1 Step 651 Train Loss: 0.4946
Epoch 1 Step 701 Train Loss: 0.4586
Epoch 1 Step 751 Train Loss: 0.4558
Epoch 1 Step 801 Train Loss: 0.4740
Epoch 1 Step 851 Train Loss: 0.4229
Epoch 1 Step 901 Train Loss: 0.5446
Epoch 1 Step 951 Train Loss: 0.4253
Epoch 1 Step 1001 Train Loss: 0.4243
Epoch 1 Step 1051 Train Loss: 0.4807
Epoch 1 Step 1101 Train Loss: 0.5051
Epoch 1 Step 1151 Train Loss: 0.4311
Epoch 1 Step 1201 Train Loss: 0.4374
Epoch 1 Step 1251 Train Loss: 0.4474
Epoch 1 Step 1301 Train Loss: 0.4794
Epoch 

2026-10-06 14:50:58,995 INFO lasser.utils: [gears_train] done in 343.7s, peak CUDA memory 0.91 GB
2026-10-06 14:50:59,020 INFO lasser.utils: [evaluate] start
2026-10-06 14:54:02,662 INFO lasser.evaluation: test (eval.py): pearson_delta=0.4555 mse_top20_de=0.2530 nmse_top20_de=1.4229 over 107 perts
2026-10-06 14:54:02,663 INFO lasser.evaluation:   combo_seen0    pearson_delta=0.5222 mse_top20_de=0.2543 (9 perts)
2026-10-06 14:54:02,664 INFO lasser.evaluation:   combo_seen1    pearson_delta=0.4770 mse_top20_de=0.2791 (43 perts)
2026-10-06 14:54:02,664 INFO lasser.evaluation:   combo_seen2    pearson_delta=0.5230 mse_top20_de=0.2572 (19 perts)
2026-10-06 14:54:02,665 INFO lasser.evaluation:   unseen_single  pearson_delta=0.3775 mse_top20_de=0.2193 (36 perts)
2026-10-06 14:54:02,667 INFO lasser.utils: [evaluate] done in 183.6s, peak CUDA memory 1.19 GB
2026-10-06 14:54:02,699 INFO lasser.tracking: appended run 348393ed to /kaggle/working/runs/runs.csv


('/kaggle/working/runs/norman_seed1_gl_20261006-144430',
 {'threshold': 0.2,
  'hit_rate_added': 0.03598640859127045,
  'hit_rate_added_only': 0.03598640859127045,
  'hit_rate_topup': nan,
  'hit_rate_degree_matched_random': 0.0016478706384077668,
  'hit_rate_a0': 0.5806451439857483,
  'hit_rate_removed': 0.75,
  'passes': True},
 {'a0_edges': 6364,
  'removed': 20,
  'added_directed': 38848,
  'topup_directed': 0,
  'refined_edges': 45192,
  'new_weight_mean': 0.24376694858074188,
  'a0_weight_mean': 0.5615240931510925})

In [8]:
# 7. Results: the global run table, and any past run loaded back.
import pandas as pd
pd.set_option("display.max_columns", 200)
runs = pd.read_csv("/kaggle/working/runs/runs.csv")
cols = ["run_id", "seed", "graph_learning", "gsr_init_gene_emb", "graph_num_edges", "graph_isolated_genes",
        "test_pearson_delta", "test_mse_top20_de", "test_nmse_top20_de",
        "test_unseen_single_pearson_delta", "test_combo_seen0_pearson_delta",
        "test_combo_seen1_pearson_delta", "test_combo_seen2_pearson_delta", "run_dir"]
display(runs[[c for c in cols if c in runs.columns]])

r = load_run(gl["run_dir"])
display(r["epoch_log"])
r["test_metrics"]["paper"]["subgroups"]

,run_id,seed,graph_learning,gsr_init_gene_emb,graph_num_edges,graph_isolated_genes,test_pearson_delta,test_mse_top20_de,test_nmse_top20_de,test_unseen_single_pearson_delta,test_combo_seen0_pearson_delta,test_combo_seen1_pearson_delta,test_combo_seen2_pearson_delta,run_dir
0,87d0540f,1,False,False,6364,4419,0.360435,0.346523,3.965999,0.300777,0.392480,0.384491,0.403849,/kaggle/working/runs/norman_seed1_base_2026100...
1,348393ed,1,True,False,45192,0,0.455471,0.253001,1.422869,0.377455,0.522169,0.476969,0.523044,/kaggle/working/runs/norman_seed1_gl_20261006-...


,epoch,train_mse,val_mse,train_mse_de,val_mse_de,train_loss_logged_mean,n_logged_steps,is_best
0,1,0.0041,0.0049,0.1399,0.3268,0.481756,32,True


{'combo_seen0': {'n_perts': 9,
  'mse': 0.00699058821838763,
  'pearson': 0.9797668706798578,
  'mse_top20_de': 0.2542625818815496,
  'mse_top20_de_noperturb': 0.3686102908104658,
  'nmse_top20_de': 3.2012983379161843,
  'pearson_top20_de': 0.7282930362177019,
  'pearson_delta': 0.5221685088247261,
  'pearson_delta_top20_de': 0.40766366227369755,
  'frac_opposite_top20_de': 0.5388888888888889,
  'frac_correct_direction_top20_de': 0.46111111111111114,
  'nmse_top20_de_ratio_of_means': 0.6897869870168324},
 'combo_seen1': {'n_perts': 43,
  'mse': 0.007933117530503592,
  'pearson': 0.9778415929983577,
  'mse_top20_de': 0.2791112255911494,
  'mse_top20_de_noperturb': 0.48487508398762275,
  'nmse_top20_de': 1.625126437932775,
  'pearson_top20_de': 0.8424886669466133,
  'pearson_delta': 0.47696869114012164,
  'pearson_delta_top20_de': 0.5781047782492842,
  'frac_opposite_top20_de': 0.4011627906976744,
  'frac_correct_direction_top20_de': 0.5988372093023255,
  'nmse_top20_de_ratio_of_means': 